# Homework #2 — semantic retrieval, interactively

This notebook is a thin front-end over `scripts/rag_lib.py`. It defines **no retrieval logic of
its own** — the same functions the CLI uses are imported here, so the notebook and
`scripts/retrieval.py` can never drift apart.

Prerequisites, in order:

```bash
pip install -r requirements.txt
python scripts/prepare_knowledge_base.py   # data/processed/chunks.jsonl
python scripts/build_index.py              # index/chroma/  (needs OPENAI_API_KEY)
```

`OPENAI_API_KEY` is read from the environment or from a gitignored `.env` at the repo root.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))

from rag_lib import Settings, open_collection, read_manifest, search
from retrieval import render

settings = Settings.from_env()
manifest = read_manifest(settings)
collection = open_collection(settings)

print(f"model      : {manifest['embedding_model']}")
print(f"dimension  : {manifest['dimension']}")
print(f"chunks     : {manifest['chunk_count']}")
print(f"vectors    : {collection.count()}")
print(f"score      : {manifest['score']}")

model      : text-embedding-3-small
dimension  : 1536
chunks     : 77
vectors    : 77
score      : 1 - cosine_distance


## Ask a question

Change `QUERY` and re-run. The query is embedded with the **same** model as the chunks —
`read_manifest` refuses to proceed otherwise, so a mismatched index fails loudly instead of
quietly returning bad neighbours.

In [2]:
QUERY = "How does a freight exchange match loads to carriers?"
K = 3

hits = search(QUERY, settings, k=K, collection=collection)
print(render(QUERY, hits))

Query: How does a freight exchange match loads to carriers?

Top-1: freight_exchange_domain_primer_chunk_001 | score: 0.807
  Text: Freight Exchange Fundamentals: Actors, Loads, and Matching > What A Freight Exchange Is. A freight exchange is a two-sided digital marketplace in which one side publishes transport demand and the other offers vehicle ca…
  Source: data/raw/freight-exchange-domain-primer.md
  Document: freight_exchange_domain_primer | Section: What A Freight Exchange Is | Type: concept-guide

Top-2: freight_exchange_domain_primer_chunk_010 | score: 0.798
  Text: Freight Exchange Fundamentals: Actors, Loads, and Matching > Load Matching Mechanics. Matching is the search problem at the centre of a freight exchange, and its inputs are highly structured. Carriers filter primarily b…
  Source: data/raw/freight-exchange-domain-primer.md
  Document: freight_exchange_domain_primer | Section: Load Matching Mechanics | Type: concept-guide

Top-3: freight_exchange_domain_primer_chunk_

## The full evaluation set

The ten queries in `data/eval/test_queries.json`, scored by category. This reproduces the table in
`outputs/retrieval_examples.md`; the paraphrase row is the interesting one.

In [3]:
import statistics

from run_test_queries import load_queries, verdict

by_category: dict[str, list[float]] = {}

for entry in load_queries(REPO_ROOT / "data/eval/test_queries.json"):
    results = search(entry["query"], settings, k=3, collection=collection)
    by_category.setdefault(entry["category"], []).append(results[0].score)
    print(f"{entry['id']}  {results[0].score:.3f}  {verdict(results, entry['expected_documents'])}")

print()
for category, scores in by_category.items():
    print(f"{category:16s} n={len(scores)}  mean top-1 = {statistics.mean(scores):.3f}")

q01  0.536  hit — top-1 is from an expected document


q02  0.582  hit — top-1 is from an expected document


q03  0.685  hit — top-1 is from an expected document


q04  0.413  hit — top-1 is from an expected document


q05  0.424  hit — top-1 is from an expected document


q06  0.431  hit — top-1 is from an expected document


q07  0.602  hit — top-1 is from an expected document


q08  0.646  hit — top-1 is from an expected document


q09  0.483  hit — top-1 is from an expected document


q10  0.266  out-of-corpus — judged on score separation, not on document match

direct           n=3  mean top-1 = 0.601
paraphrase       n=3  mean top-1 = 0.423
cross-document   n=3  mean top-1 = 0.577
out-of-corpus    n=1  mean top-1 = 0.266


## Why the out-of-corpus query matters

Nothing in a logistics corpus answers a question about fine-tuning language models — yet the
retriever still returns three neatly formatted results. Only the score reveals the problem, and no
threshold is enforced anywhere in the code. This is the control a RAG system needs before an LLM
sees these chunks.

In [4]:
OFF_TOPIC = "What is the best way to fine-tune a large language model on a custom dataset?"
ON_TOPIC = "What is the strangler-fig pattern for migrating a monolith?"

off = search(OFF_TOPIC, settings, k=3, collection=collection)
on = search(ON_TOPIC, settings, k=3, collection=collection)

print(f"off-topic top-1 : {off[0].score:.3f}  -> {off[0].chunk_id}")
print(f"on-topic  top-1 : {on[0].score:.3f}  -> {on[0].chunk_id}")
print(f"\nseparation      : {on[0].score - off[0].score:.3f}")
print("\nBoth queries returned 3 results. Only the score distinguishes them.")

off-topic top-1 : 0.266  -> monolith_to_microservices_migration_chunk_014
on-topic  top-1 : 0.685  -> monolith_to_microservices_migration_chunk_010

separation      : 0.419

Both queries returned 3 results. Only the score distinguishes them.
